In [2]:
import pandas as pd
import numpy as np
import warnings

# Suppress warnings for a cleaner output
warnings.filterwarnings('ignore')

print("--- Task 2: Final Cleanup and Soil Data Integration ---")

# --- Part A: Final Base Data Cleanup ---

print("\n--- Part A: Loading and Cleaning base_data_v1.csv ---")

try:
    # 1. Load Data
    df = pd.read_csv('base_data_v1.csv')
    print("Loaded 'base_data_v1.csv' successfully.")
except FileNotFoundError:
    print("ERROR: 'base_data_v1.csv' not found.")
    print("Please make sure the file from Task 1 is in the same directory.")
    # Exit or raise error
    # For this script, we'll create a dummy df to show the error
    df = pd.DataFrame() 

if not df.empty:
    # 2. Handle 'year' Column
    print("Converting 'year' column from 'YYYY-YY' to 'YYYY' integer...")
    df['year'] = df['year'].apply(lambda x: int(x.split('-')[0]))

    # 3. Drop Redundant & Leaky Columns
    print("Dropping leaky and redundant columns (production, state, etc.)...")
    columns_to_drop = ['production', 'state', 'area_units', 'production_units']
    df = df.drop(columns=columns_to_drop)

    # 4. Standardize District Names
    print("Standardizing 'district' column to lowercase...")
    df['district'] = df['district'].str.lower()
    
    print("Base data cleaned.")
    print(f"DataFrame shape after Part A: {df.shape}")


# --- Part B: Integrate Categorical Soil Type (Table 2) ---

print("\n--- Part B: Creating and Merging Categorical Soil Type (Table 2) ---")

# 1. Create soil_type_df (Data from your research doc )
soil_type_data = {
    'district': ['Angul', 'Balasore', 'Bargarh', 'Bhadrak', 'Bolangir', 'Boudh', 
                   'Cuttack', 'Deogarh', 'Dhenkanal', 'Gajapati', 'Ganjam', 
                   'Jagatsinghpur', 'Jajpur', 'Jharsuguda', 'Kalahandi', 'Kandhamal', 
                   'Kendrapara', 'Keonjhar', 'Khordha', 'Koraput', 'Malkangiri', 
                   'Mayurbhanj', 'Nabarangpur', 'Nayagarh', 'Nuapada', 'Puri', 
                   'Rayagada', 'Sambalpur', 'Sonepur', 'Sundargarh'],
    'primary_soil_type': ['Black Soil', 'Coastal Salt Affected Alluvial, Deltaic Alluvial',
                          'Mixed Red & Yellow, Black, Mixed Red & Black', 
                          'Coastal Salt Affected Alluvial, Deltaic Alluvial',
                          'Red, Black, Mixed Red & Black', 'Black Soil', 
                          'Laterite, Deltaic Alluvial', 'Mixed Red & Yellow',
                          'Red, Laterite', 'Deltaic Alluvial',
                          'Red, Coastal Salt Affected Alluvial, Deltaic Alluvial, Black, Brown Forest',
                          'Coastal Salt Affected Alluvial, Deltaic Alluvial', 'Deltaic Alluvial',
                          np.nan,
                          'Red, Black', 'Brown Forest', 
                          'Coastal Salt Affected Alluvial, Deltaic Alluvial',
                          'Red, Laterite', 'Laterite, Coastal Salt Affected Alluvial', 'Red',
                          'Red, Black', 'Red, Laterite', 'Red', 'Laterite, Brown Forest',
                          'Red, Black', 'Laterite, Coastal Salt Affected Alluvial, Deltaic Alluvial, Black',
                          'Red, Brown Forest', 'Laterite, Red & Yellow, Black, Mixed Red & Black',
                          'Black, Mixed Red & Black', 'Mixed Red & Yellow']
}

soil_type_df = pd.DataFrame(soil_type_data)

# Standardize district column to lowercase
soil_type_df['district'] = soil_type_df['district'].str.lower()
print("Created 'soil_type_df'.")

# 2. Merge Soil Type
print("Merging 'soil_type_df' into main DataFrame...")
df = pd.merge(df, soil_type_df, on='district', how='left')

print(f"DataFrame shape after Part B: {df.shape}")
print("Categorical soil data merged.")


# --- Part C: Integrate Numerical Soil Nutrients (Table 1) ---

print("\n--- Part C: Creating, Cleaning, and Merging Numerical Soil Nutrients (Table 1) ---")

# 1. Create soil_nutrient_raw_df (Data from your research doc )
nutrient_data = [
    ('Balasore', 'Bhograi, Jaleswar, Sadar (Range)', '5.5 - 6.5', '106.0 - 172.47', '4.75 - 23.48', '77.50 - 244.21'),
    ('Cuttack', 'Athagarh', '5.31', '177.45', '24.90', '208.32'),
    ('Cuttack', 'Badamba', '6.07', '150.30', '5.58', '232.53'),
    ('Cuttack', 'Banki', '5.24', '147.30', '10.52', '278.04'),
    ('Cuttack', 'Banki–Dampara', '4.95', '169.06', '15.80', '440.62'),
    ('Cuttack', 'Barang', '5.26', '161.70', '27.35', '158.48'),
    ('Cuttack', 'Cuttack-Sadar', '5.41', '145.20', '8.33', '194.58'),
    ('Cuttack', 'Kantapada', '5.51', '150.26', '15.79', '233.40'),
    ('Cuttack', 'Mahanga', '5.31', '181.90', '3.81', '161.09'),
    ('Cuttack', 'Narasinghpur', '6.22', '154.70', '8.32', '241.40'),
    ('Cuttack', 'Niali', '5.17', '162.60', '7.02', '185.09'),
    ('Cuttack', 'Nischintakoili', '5.54', '202.20', '5.66', '160.60'),
    ('Cuttack', 'Salepur', '5.46', '191.80', '8.77', '303.25'),
    ('Cuttack', 'Tangi-Choudwar', '5.19', '173.09', '4.74', '103.03'),
    ('Cuttack', 'Tigiria', '5.68', '164.70', '11.21', '161.70'),
    ('Ganjam', 'Jiliba Village (Range)', '5.5 - 7.5', '87.8 - 150.5', '4.7 - 198.2', '54.6 - 435.6'),
    ('Khordha', 'High Land (Mean)', '4.96', np.nan, '8.84', np.nan),
    ('Khordha', 'Medium Land (Mean)', '5.39', np.nan, '11.74', np.nan),
    ('Khordha', 'Low Land (Mean)', '5.79', np.nan, '15.45', np.nan),
    ('Puri', 'Astarang', '5.78', '178.00', '25.15', '412.00'),
    ('Puri', 'Bramhagiri', '5.52', '204.00', '17.54', '389.00'),
    ('Puri', 'Delanga', '5.48', '212.00', '22.48', '214.00'),
    ('Puri', 'Gop', '5.42', '216.00', '20.16', '201.00'),
    ('Puri', 'Kanas', '5.68', '221.00', '19.85', '198.00'),
    ('Puri', 'Kakatapur', '5.27', '242.00', '18.74', '185.00'),
    ('Puri', 'Krishna Prasad', '5.64', '201.00', '15.42', '225.00'),
    ('Puri', 'Nimapada', '5.31', '198.00', '24.18', '196.00'),
    ('Puri', 'Pipili', '5.34', '225.00', '21.14', '165.00'),
    ('Puri', 'Puri Sadar', '5.45', '235.00', '20.15', '175.00'),
    ('Puri', 'Satyabadi', '5.72', '245.00', '23.16', '205.00')
]

soil_nutrient_raw_df = pd.DataFrame(nutrient_data, columns=[
    'district', 'sub_region_block', 'mean_ph', 'mean_n', 'mean_p', 'mean_k'
])
print("Created 'soil_nutrient_raw_df'.")

# 2. Define Helper Function
def convert_range_to_mean(x):
    if isinstance(x, str):
        if '-' in x:
            parts = x.split('-')
            try:
                # Clean parts of any extra whitespace
                low = float(parts[0].strip())
                high = float(parts[1].strip())
                return (low + high) / 2
            except ValueError:
                return np.nan
        else:
            try:
                return float(x)
            except ValueError:
                return np.nan
    return x
print("Defined 'convert_range_to_mean' helper function.")

# 3. Clean 'soil_nutrient_raw_df'
print("Cleaning nutrient data (applying range converter)...")
nutrient_cols = ['mean_ph', 'mean_n', 'mean_p', 'mean_k']
for col in nutrient_cols:
    soil_nutrient_raw_df[col] = soil_nutrient_raw_df[col].apply(convert_range_to_mean)

# Standardize district column to lowercase
soil_nutrient_raw_df['district'] = soil_nutrient_raw_df['district'].str.lower()
print("Nutrient data cleaned.")

# 4. Aggregate to District Level
print("Aggregating nutrient data from block/sub-region to district level...")
soil_nutrient_agg_df = soil_nutrient_raw_df.groupby('district')[nutrient_cols].mean().reset_index()

print("Nutrient data aggregated. Head of aggregated data:")
print(soil_nutrient_agg_df.head())

# 5. Merge Nutrient Data
print("\nMerging aggregated nutrient data into main DataFrame...")
df = pd.merge(df, soil_nutrient_agg_df, on='district', how='left')
print("Numerical soil nutrient data merged.")


# --- Part D: Submit for Review ---

print("\n--- Part D: Final Review of Merged DataFrame ---")

print("\n--- Final DataFrame Info ---")
df.info()

print("\n--- Final DataFrame Head ---")
print(df.head())

print("\n--- Final DataFrame Null Check (Post-Merge) ---")
# Calculate nulls as a percentage for better context
null_counts = df.isnull().sum()
null_percent = (null_counts / len(df)) * 100
null_report = pd.DataFrame({'null_count': null_counts, 'null_percentage': null_percent})
print(null_report)

# 3. Save the new file
try:
    df.to_csv('base_data_v2_with_soil.csv', index=False)
    print(f"\n✅ SUCCESS: Final dataset saved as 'base_data_v2_with_soil.csv'")
except Exception as e:
    print(f"\n❌ ERROR: Could not save final file. Reason: {e}")

--- Task 2: Final Cleanup and Soil Data Integration ---

--- Part A: Loading and Cleaning base_data_v1.csv ---
Loaded 'base_data_v1.csv' successfully.
Converting 'year' column from 'YYYY-YY' to 'YYYY' integer...
Dropping leaky and redundant columns (production, state, etc.)...
Standardizing 'district' column to lowercase...
Base data cleaned.
DataFrame shape after Part A: (16153, 8)

--- Part B: Creating and Merging Categorical Soil Type (Table 2) ---
Created 'soil_type_df'.
Merging 'soil_type_df' into main DataFrame...
DataFrame shape after Part B: (16153, 9)
Categorical soil data merged.

--- Part C: Creating, Cleaning, and Merging Numerical Soil Nutrients (Table 1) ---
Created 'soil_nutrient_raw_df'.
Defined 'convert_range_to_mean' helper function.
Cleaning nutrient data (applying range converter)...
Nutrient data cleaned.
Aggregating nutrient data from block/sub-region to district level...
Nutrient data aggregated. Head of aggregated data:
   district   mean_ph      mean_n      mea

In [3]:
import pandas as pd

# Load the file we just created
df = pd.read_csv('base_data_v2_with_soil.csv')

# Get the "ground truth" spellings from the data
unique_districts = sorted(df['district'].unique())

print(f"Ground Truth District Spellings ({len(unique_districts)}):")
print(unique_districts)

Ground Truth District Spellings (30):
['anugul', 'balangir', 'baleshwar', 'bargarh', 'bhadrak', 'boudh', 'cuttack', 'deogarh', 'dhenkanal', 'gajapati', 'ganjam', 'jagatsinghapur', 'jajapur', 'jharsuguda', 'kalahandi', 'kandhamal', 'kendrapara', 'kendujhar', 'khordha', 'koraput', 'malkangiri', 'mayurbhanj', 'nabarangpur', 'nayagarh', 'nuapada', 'puri', 'rayagada', 'sambalpur', 'sonepur', 'sundargarh']


In [4]:
import pandas as pd
import numpy as np

print("--- Task 2.1: Fixing Soil Merge Integrity ---")

# --- Part A: Load and Clean base_data_v1.csv ---
print("\n--- Part A: Loading and Cleaning base_data_v1.csv ---")
try:
    df = pd.read_csv('base_data_v1.csv')
    print("Loaded 'base_data_v1.csv' successfully.")
except FileNotFoundError:
    print("ERROR: 'base_data_v1.csv' not found. Stopping.")
    # In a real script, you'd raise an error.
    # For this, we'll stop execution by creating an empty df.
    df = pd.DataFrame()

if not df.empty:
    # Perform all Part A cleanup steps again
    df['year'] = df['year'].apply(lambda x: int(x.split('-')[0]))
    columns_to_drop = ['production', 'state', 'area_units', 'production_units']
    df = df.drop(columns=columns_to_drop)
    df['district'] = df['district'].str.lower()
    
    # VERIFY that our ground truth list matches the df's list
    unique_districts_in_df = sorted(df['district'].unique())
    print("Cleaned base data. District spellings verified.")


# --- Part B: Integrate Categorical Soil Type (CORRECTED) ---
print("\n--- Part B: Creating and Merging CORRECTED Categorical Soil Type ---")

# 1. Create soil_type_df with GROUND TRUTH spellings
soil_type_data = {
    'district': [
        'anugul', 'baleshwar', 'bargarh', 'bhadrak', 'balangir', 'boudh', 
        'cuttack', 'deogarh', 'dhenkanal', 'gajapati', 'ganjam', 
        'jagatsinghapur', 'jajapur', 'jharsuguda', 'kalahandi', 'kandhamal', 
        'kendrapara', 'kendujhar', 'khordha', 'koraput', 'malkangiri', 
        'mayurbhanj', 'nabarangpur', 'nayagarh', 'nuapada', 'puri', 
        'rayagada', 'sambalpur', 'sonepur', 'sundargarh'
    ],
    'primary_soil_type': [
        'Black Soil', 'Coastal Salt Affected Alluvial, Deltaic Alluvial', # baleshwar
        'Mixed Red & Yellow, Black, Mixed Red & Black', 
        'Coastal Salt Affected Alluvial, Deltaic Alluvial',
        'Red, Black, Mixed Red & Black', 'Black Soil', 
        'Laterite, Deltaic Alluvial', 'Mixed Red & Yellow',
        'Red, Laterite', 'Deltaic Alluvial',
        'Red, Coastal Salt Affected Alluvial, Deltaic Alluvial, Black, Brown Forest',
        'Coastal Salt Affected Alluvial, Deltaic Alluvial', 'Deltaic Alluvial', # jajapur
        np.nan, # 'Data Not Explicitly Provided in Sources'
        'Red, Black', 'Brown Forest', 
        'Coastal Salt Affected Alluvial, Deltaic Alluvial',
        'Red, Laterite', 'Laterite, Coastal Salt Affected Alluvial', 'Red', # kendujhar
        'Red, Black', 'Red, Laterite', 'Red', 'Laterite, Brown Forest',
        'Red, Black', 'Laterite, Coastal Salt Affected Alluvial, Deltaic Alluvial, Black',
        'Red, Brown Forest', 'Laterite, Red & Yellow, Black, Mixed Red & Black',
        'Black, Mixed Red & Black', 'Mixed Red & Yellow'
    ]
}

soil_type_df = pd.DataFrame(soil_type_data)
# No need to lowercase, we typed it directly.
print("Created corrected 'soil_type_df'.")

# 2. Merge Soil Type
df = pd.merge(df, soil_type_df, on='district', how='left')
print("Categorical soil data merged successfully.")


# --- Part C: Integrate Numerical Soil Nutrients (CORRECTED) ---
print("\n--- Part C: Creating and Merging CORRECTED Numerical Soil Nutrients ---")

# 1. Create soil_nutrient_raw_df with GROUND TRUTH spellings
nutrient_data = [
    # Using 'baleshwar', 'cuttack', 'ganjam', 'khordha', 'puri'
    ('baleshwar', 'Bhograi, Jaleswar, Sadar (Range)', '5.5 - 6.5', '106.0 - 172.47', '4.75 - 23.48', '77.50 - 244.21'),
    ('cuttack', 'Athagarh', '5.31', '177.45', '24.90', '208.32'),
    ('cuttack', 'Badamba', '6.07', '150.30', '5.58', '232.53'),
    ('cuttack', 'Banki', '5.24', '147.30', '10.52', '278.04'),
    ('cuttack', 'Banki–Dampara', '4.95', '169.06', '15.80', '440.62'),
    ('cuttack', 'Barang', '5.26', '161.70', '27.35', '158.48'),
    ('cuttack', 'Cuttack-Sadar', '5.41', '145.20', '8.33', '194.58'),
    ('cuttack', 'Kantapada', '5.51', '150.26', '15.79', '233.40'),
    ('cuttack', 'Mahanga', '5.31', '181.90', '3.81', '161.09'),
    ('cuttack', 'Narasinghpur', '6.22', '154.70', '8.32', '241.40'),
    ('cuttack', 'Niali', '5.17', '162.60', '7.02', '185.09'),
    ('cuttack', 'Nischintakoili', '5.54', '202.20', '5.66', '160.60'),
    ('cuttack', 'Salepur', '5.46', '191.80', '8.77', '303.25'),
    ('cuttack', 'Tangi-Choudwar', '5.19', '173.09', '4.74', '103.03'),
    ('cuttack', 'Tigiria', '5.68', '164.70', '11.21', '161.70'),
    ('ganjam', 'Jiliba Village (Range)', '5.5 - 7.5', '87.8 - 150.5', '4.7 - 198.2', '54.6 - 435.6'),
    ('khordha', 'High Land (Mean)', '4.96', np.nan, '8.84', np.nan),
    ('khordha', 'Medium Land (Mean)', '5.39', np.nan, '11.74', np.nan),
    ('khordha', 'Low Land (Mean)', '5.79', np.nan, '15.45', np.nan),
    ('puri', 'Astarang', '5.78', '178.00', '25.15', '412.00'),
    ('puri', 'Bramhagiri', '5.52', '204.00', '17.54', '389.00'),
    ('puri', 'Delanga', '5.48', '212.00', '22.48', '214.00'),
    ('puri', 'Gop', '5.42', '216.00', '20.16', '201.00'),
    ('puri', 'Kanas', '5.68', '221.00', '19.85', '198.00'),
    ('puri', 'Kakatapur', '5.27', '242.00', '18.74', '185.00'),
    ('puri', 'Krishna Prasad', '5.64', '201.00', '15.42', '225.00'),
    ('puri', 'Nimapada', '5.31', '198.00', '24.18', '196.00'),
    ('puri', 'Pipili', '5.34', '225.00', '21.14', '165.00'),
    ('puri', 'Puri Sadar', '5.45', '235.00', '20.15', '175.00'),
    ('puri', 'Satyabadi', '5.72', '245.00', '23.16', '205.00')
]

soil_nutrient_raw_df = pd.DataFrame(nutrient_data, columns=[
    'district', 'sub_region_block', 'mean_ph', 'mean_n', 'mean_p', 'mean_k'
])
print("Created corrected 'soil_nutrient_raw_df'.")

# 2. Define Helper Function
def convert_range_to_mean(x):
    if isinstance(x, str):
        if '-' in x:
            parts = x.split('-')
            try:
                low = float(parts[0].strip())
                high = float(parts[1].strip())
                return (low + high) / 2
            except ValueError: return np.nan
        else:
            try: return float(x)
            except ValueError: return np.nan
    return x

# 3. Clean 'soil_nutrient_raw_df'
nutrient_cols = ['mean_ph', 'mean_n', 'mean_p', 'mean_k']
for col in nutrient_cols:
    soil_nutrient_raw_df[col] = soil_nutrient_raw_df[col].apply(convert_range_to_mean)
# No need to lowercase district, it's already correct.

# 4. Aggregate to District Level
soil_nutrient_agg_df = soil_nutrient_raw_df.groupby('district')[nutrient_cols].mean().reset_index()
print("Nutrient data aggregated.")

# 5. Merge Nutrient Data
df = pd.merge(df, soil_nutrient_agg_df, on='district', how='left')
print("Numerical soil nutrient data merged successfully.")


# --- Part D: Submit for Review ---
print("\n--- Part D: Final Review of CORRECTED Merged DataFrame ---")

print("\n--- Final DataFrame Info ---")
df.info()

print("\n--- Final DataFrame Head ---")
print(df.head())

print("\n--- Final DataFrame Null Check (Post-Merge) ---")
null_counts = df.isnull().sum()
null_percent = (null_counts / len(df)) * 100
null_report = pd.DataFrame({'null_count': null_counts, 'null_percentage': null_percent})
print(null_report)

# 3. Save the new, corrected file
try:
    df.to_csv('base_data_v3_soil_corrected.csv', index=False)
    print(f"\n✅ SUCCESS: Corrected dataset saved as 'base_data_v3_soil_corrected.csv'")
except Exception as e:
    print(f"\n❌ ERROR: Could not save final file. Reason: {e}")

--- Task 2.1: Fixing Soil Merge Integrity ---

--- Part A: Loading and Cleaning base_data_v1.csv ---
Loaded 'base_data_v1.csv' successfully.
Cleaned base data. District spellings verified.

--- Part B: Creating and Merging CORRECTED Categorical Soil Type ---
Created corrected 'soil_type_df'.
Categorical soil data merged successfully.

--- Part C: Creating and Merging CORRECTED Numerical Soil Nutrients ---
Created corrected 'soil_nutrient_raw_df'.
Nutrient data aggregated.
Numerical soil nutrient data merged successfully.

--- Part D: Final Review of CORRECTED Merged DataFrame ---

--- Final DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16153 entries, 0 to 16152
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   district           16153 non-null  object 
 1   crop               16153 non-null  object 
 2   year               16153 non-null  int64  
 3   season             16153 non-